# Nettoyage de l'export hebdomadaire
**Responsable : Ihman** · export `Pharma_Ventes_Weekly.csv` · module `src/pharma/cleaning/weekly.py`

**Pourquoi ce nettoyage ?** Comme le journalier, l'export hebdomadaire sert de **contrôle** face à l'horaire (la référence).
Il a deux particularités : chaque ligne est datée du **dimanche** qui termine la semaine (lundi → dimanche), et la
première et la dernière semaine sont coupées par le début et la fin de l'export. Les ventes retenues sont les
**semaines complètes recalculées depuis l'horaire** ; elles servent notamment à l'analyse de la grippe, qui est hebdomadaire.

## 0 · Préparer le notebook

In [ ]:
import sys
from pathlib import Path

# Le notebook est dans notebooks/ : on remonte à la racine du projet pour importer le code
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from pharma import config
from pharma.cleaning import checks, hourly, loading

G = config.GROUP_CODES          # les 8 groupes ATC, dans l'ordre des colonnes

## 1 · Lire l'export brut

In [ ]:
brut = pd.read_csv(config.RAW_DIR / config.EXPORTS["weekly"])
print(brut.shape)
brut.head()            # seulement datum + les 8 groupes : pas de colonnes ajoutées par le logiciel

In [ ]:
export = loading.load_export("weekly")
derived = loading.load_derived_columns("weekly")       # vide : rien à contrôler de ce côté
print(f"{len(export)} semaines, du {export.index.min():%d/%m/%Y} au {export.index.max():%d/%m/%Y}")
export.head()

## 2 · Les contrôles communs (1 à 4)

In [ ]:
journal = []
checks.scope(journal, "weekly", export)                                  # 1 · périmètre
export = checks.duplicates_and_negatives(journal, "weekly", export)     # 2, 3, 4 · doublons, manquants, négatifs
pd.DataFrame(journal)

## 3 · Chaque semaine est-elle datée d'un dimanche ?

In [ ]:
# dayofweek numérote les jours de 0 (lundi) à 6 (dimanche)
export.index.dayofweek.value_counts()

## 4 · Continuité : 7 jours entre deux semaines ? (contrôle 5)

In [ ]:
ecarts_en_jours = export.index.to_series().diff().dt.days      # temps écoulé depuis la semaine précédente
ecarts_en_jours.value_counts()          # tout doit valoir 7 (la première ligne n'a pas de précédente)

## 5 · L'hebdomadaire concorde-t-il avec l'horaire ? (contrôle 9)

In [ ]:
# La référence : l'export horaire nettoyé par Adelaida (hourly.py)
horaire = hourly.clean_hourly(loading.load_export("hourly"), loading.load_derived_columns("hourly"), [])
daily_all = horaire.daily_all           # ventes par jour recalculées depuis l'horaire (tous les jours)
complete_day = horaire.complete_day     # True si le jour est complet selon l'horaire
print(f"{len(daily_all)} jours recalculés, dont {int(complete_day.sum())} complets")

In [ ]:
# On additionne les jours de l'horaire par semaine finissant le dimanche (W-SUN = Week, SUNday)
semaines_recalculees = daily_all.resample("W-SUN").sum()
ecarts = checks.compare(export, semaines_recalculees, "weekly", "%d/%m/%Y")
print(f"{len(ecarts)} case(s) en écart sur {len(export) * len(G)}")
ecarts

## 6 · Semaines incomplètes : exclues des ventes propres (contrôle 7)

In [ ]:
# Nombre de jours COMPLETS (selon l'horaire) dans chaque semaine : une semaine complète en a 7
jours_complets = complete_day.resample("W-SUN").sum()
jours_complets[jours_complets < 7]

- La **première** semaine (finissant le 05/01/2014) n'a que **4 jours** : l'export commence le jeudi 02/01.
- La **dernière** (au 13/10/2019) n'a qu'**1 jour complet** : le lundi 07/10 ; le 08/10 s'arrête à 19 h.

Garder ces semaines ferait croire à une chute des ventes : elles sont exclues.

In [ ]:
ventes = semaines_recalculees.loc[jours_complets[jours_complets == 7].index, G]
print(f"Ventes propres : {len(ventes)} semaines complètes, du {ventes.index.min():%d/%m/%Y} au {ventes.index.max():%d/%m/%Y}")

## Étape finale · Le module `weekly.py`, tel que `main.py` l'utilise
Contrat : `clean_weekly(export, derived, daily_all, complete_day, rows) -> (weekly, ecarts)`.

In [ ]:
from pharma.cleaning.weekly import clean_weekly

journal_final = []
weekly, ecarts_final = clean_weekly(loading.load_export("weekly"), derived, daily_all, complete_day, journal_final)
print(f"{len(weekly)} semaines complètes ; {len(ecarts_final)} écart avec l'horaire")
pd.DataFrame(journal_final)

## Ce que je retiens

**Constat 1 · Un fichier propre.** 302 semaines, du 05/01/2014 au 13/10/2019, toutes datées d'un dimanche et espacées
de 7 jours exactement. Aucun doublon, aucune case vide, aucune vente négative.

**Constat 2 · Il concorde avec l'horaire.** Chaque semaine égale la somme de ses jours (0 case en écart) : l'hebdomadaire est
fiable, mais on ne l'utilise que comme contrôle.

**Constat 3 · Deux semaines incomplètes, exclues.** La première n'a que 4 jours (début le jeudi 02/01/2014) — ce qui n'était
pas noté au départ — et la dernière 1 seul jour complet. **300 semaines complètes** retenues, du 12/01/2014 au 06/10/2019.

**Utilité.** Ces semaines complètes sont alignées sur l'incidence de la grippe du réseau Sentinelles (semaines ISO, du lundi au
dimanche) dans `analysis/environment.py`.